In [3]:
import cv2
import mediapipe as mp
import joblib
import numpy as np
import pyttsx3
import time

In [4]:



model = joblib.load("sign_language_model.pkl")


def speak_letter(text):
    engine = pyttsx3.init()
    engine.setProperty('rate', 160)
    engine.setProperty('volume', 1.0)
    engine.say(text)
    engine.runAndWait()
    engine.stop()


mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)


sentence = ""
last_letter = ""
last_spoken_time = 0
SPEAK_DELAY = 1.2  


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            if len(landmarks) == 63:
                X = np.array(landmarks).reshape(1, -1)
                letter = model.predict(X)[0]

                current_time = time.time()

               
                if letter != last_letter and (current_time - last_spoken_time) > SPEAK_DELAY:

                    if letter.lower() == "space":
                        sentence += " "
                        speak_letter("space")

                    elif letter.lower() == "del":
                        sentence = sentence[:-1]
                        speak_letter("delete")

                    else:
                        sentence += letter
                        speak_letter(letter)

                    last_letter = letter
                    last_spoken_time = current_time

                cv2.putText(frame, f"Detected: {letter}", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

   
    cv2.rectangle(frame, (10, 420), (1200, 470), (0, 0, 0), -1)
    cv2.putText(frame, sentence, (20, 455),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("Real-Time Sign Language (Auto Speak)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


d:\vs_code_DL\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\vs_code_DL\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\vs_code_DL\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\vs_code_DL\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\vs_code_DL\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\vs_code_DL\venv\Lib\si